# Entraînement YOLO — validation croisée par patient
Six cellules de code, à exécuter dans l’ordre depuis le dossier de ce notebook.

**Préparation du PC RTX** : utiliser le venv validé par `check_rtx_setup.py`, avec `ultralytics`, `torch`, `torchvision`, `numpy`, `opencv-python`, `matplotlib`, `pyyaml`, `pandas` et `ipykernel`. Aucun paquet n’est installé par ce notebook.

**Fichiers nécessaires** : `train_config.yaml`, `kfold_train_reviewed.csv`, et `optic_disk_kfold_reviewed_dataset/` avec images, labels, masks et les cinq dossiers `folds/fold_N/` contenant train.txt, val.txt, dataset.yaml. Conserver les chemins des TXT relatifs au dossier du notebook.

**Poids préentraînés** : `yolo11n-seg.pt` est téléchargé automatiquement par Ultralytics au premier chargement (connexion Internet nécessaire). Pour travailler hors ligne, télécharger ce fichier officiel sur un PC connecté et le placer dans `models/`. Ne pas utiliser `yolo11n.pt`, qui correspond à la détection seule. Référence : https://docs.ultralytics.com/models/yolo11/ .

La cellule 4 lance réellement les cinq entraînements. Les cellules précédentes vérifient l’environnement et les données. Aucun entraînement n’a été exécuté lors de la création de ce notebook.

**Dataset reviewed** : toutes les vues doivent avoir `Matthis CHECK = done`, indépendamment de leur ancien `ANNOTATION_STATUS`. Les masques 2D ou RGB à trois canaux identiques sont acceptés. Les folds sont préparés dans `new_dataset.ipynb` : conserver leurs TXT et YAML lors du transfert sur le PC d'entraînement. Les chemins des TXT restent relatifs à la racine du projet, sans chemin propre à ce PC.


## 1 — Imports et GPU

In [ ]:
from pathlib import Path
from collections import Counter
import gc
import json
import platform
import sys
import time
import importlib.metadata

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import yaml
from IPython.display import display
from ultralytics import YOLO

print("Python :", sys.executable)
print("PyTorch :", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA indisponible : sélectionner le venv RTX validé avant de continuer.")
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "Go")


## 2 — Configuration
Changer MODEL_NAME et EXPERIMENT pour comparer une autre taille. Chaque expérience utilise un nouveau dossier ; aucune reprise automatique ni écrasement.

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / "train_kfold.ipynb").is_file():
    raise RuntimeError("Ouvrir le notebook avec son dossier comme répertoire de travail.")
MODEL_NAME = "yolo26n-seg.pt"  # alternatives : yolo11s-seg.pt, yolo11m-seg.pt
EXPERIMENT = "yolo26n_seg_512_seed42_reviewed_v1"
local_weights = ROOT / "models" / MODEL_NAME
INITIAL_WEIGHTS = str(local_weights) if local_weights.is_file() else MODEL_NAME
CONFIG = ROOT / "train_config.yaml"
DATASET = ROOT / "optic_disk_kfold_reviewed_dataset"
RUN_ROOT = ROOT / "runs" / "kfold" / EXPERIMENT
FOLD_YAMLS = [DATASET / "folds" / f"fold_{i}" / "dataset.yaml" for i in range(1, 6)]
with CONFIG.open(encoding="utf-8") as stream:
    settings = yaml.safe_load(stream)
# Explicit runtime overrides shared by every fold.
RUNTIME = dict(device=0, batch=8, workers=0, cache=False, amp=True)
print("Poids initiaux :", INITIAL_WEIGHTS)
print("Paramètres :", {**settings, **RUNTIME})
print("Résultats :", RUN_ROOT)


## 3 — Vérification des données et des patients
Vérifie les cinq partitions, les fichiers associés et la structure des labels polygonaux. Aucune écriture à cette étape.

In [ ]:
metadata = pd.read_csv(ROOT / "kfold_train_reviewed.csv", dtype=str, keep_default_na=False)
required = {"NAME", "PATIENT", "SIDE", "Matthis CHECK"}
if not required.issubset(metadata.columns):
    raise ValueError("Colonnes manquantes dans le CSV.")
for column in required:
    metadata[column] = metadata[column].str.strip()
if metadata.empty or metadata.NAME.str.lower().duplicated().any():
    raise ValueError("CSV vide ou noms d’images dupliqués.")
# The review supersedes the original annotation status; keep every done view.
if not metadata["Matthis CHECK"].str.lower().eq("done").all():
    raise ValueError("Le CSV contient des prises de vue non marquées done dans Matthis CHECK.")
if metadata.PATIENT.eq("").any():
    raise ValueError("Identifiant patient manquant.")
metadata["PATIENT"] = metadata.PATIENT.str.upper()
if metadata.duplicated(["PATIENT", "SIDE"]).any() or not metadata.SIDE.isin(["L", "R"]).all():
    raise ValueError("Œil dupliqué ou côté invalide.")
expected = {(DATASET / "images" / name).resolve() for name in metadata.NAME}
actual = {p.resolve() for p in (DATASET / "images").iterdir() if p.is_file()}
if actual != expected:
    raise ValueError("Le CSV et le dossier images ne correspondent pas exactement.")
patient_for = {(DATASET / "images" / row.NAME).resolve(): row.PATIENT for row in metadata.itertuples()}
for image in sorted(expected):
    mask = DATASET / "masks" / f"{image.stem}_mask.png"
    label = DATASET / "labels" / f"{image.stem}.txt"
    if not image.is_file() or not mask.is_file() or not label.is_file():
        raise FileNotFoundError(f"Image, masque ou label manquant : {image.name}")
    lines = label.read_text(encoding="utf-8-sig").splitlines()
    if len(lines) != 1:
        raise ValueError(f"Un seul polygone attendu : {label.name}")
    values = np.asarray([float(v) for v in lines[0].split()])
    if len(values) < 7 or (len(values) - 1) % 2 or values[0] != 0:
        raise ValueError(f"Format de label incorrect : {label.name}")
    if not np.isfinite(values).all() or np.any((values[1:] < 0) | (values[1:] > 1)):
        raise ValueError(f"Coordonnées incorrectes : {label.name}")
    im = cv2.imread(str(image), cv2.IMREAD_UNCHANGED)
    mk = cv2.imread(str(mask), cv2.IMREAD_UNCHANGED)
    if im is None or mk is None:
        raise ValueError(f"Image ou masque illisible : {image.name}")
    # Reviewed grayscale masks may be stored as RGB with identical channels.
    # Extract one channel without changing pixel values; reject true color masks.
    if mk.ndim == 3 and mk.shape[2] == 3:
        if (np.array_equal(mk[:, :, 0], mk[:, :, 1])
                and np.array_equal(mk[:, :, 0], mk[:, :, 2])):
            mk = mk[:, :, 0]
    if mk.ndim != 2 or im.shape[:2] != mk.shape:
        raise ValueError(f"Masque non gris ou dimensions incompatibles : {image.name}")

splits = {}
validation_counts = Counter()
summary = []
for fold, descriptor in enumerate(FOLD_YAMLS, 1):
    with descriptor.open(encoding="utf-8") as stream:
        data = yaml.safe_load(stream)
    if "path" in data or data.get("names") != {0: "optic_disc"}:
        raise ValueError(f"YAML inattendu : {descriptor}")
    fold_sets = {}
    for split in ("train", "val"):
        listing = descriptor.parent / data[split]
        entries = listing.read_text(encoding="utf-8-sig").splitlines()
        if not entries or any(not entry.strip() or Path(entry).is_absolute() or entry.startswith("./") for entry in entries):
            raise ValueError(f"Liste vide ou chemins non conformes à la racine du projet : {listing}")
        paths = [(ROOT / entry).resolve() for entry in entries]
        if len(set(paths)) != len(paths) or not set(paths) <= expected:
            raise ValueError(f"Doublons ou images inconnues : {listing}")
        fold_sets[split] = set(paths)
    train, val = fold_sets["train"], fold_sets["val"]
    train_patients = {patient_for[p] for p in train}
    val_patients = {patient_for[p] for p in val}
    if train & val or train | val != expected or train_patients & val_patients:
        raise ValueError(f"Fuite de patients ou partition incorrecte : fold {fold}")
    validation_counts.update(val)
    splits[fold] = fold_sets
    summary.append(dict(fold=fold, train_images=len(train), val_images=len(val),
                        train_patients=len(train_patients), val_patients=len(val_patients)))
if set(validation_counts) != expected or any(n != 1 for n in validation_counts.values()):
    raise ValueError("Chaque image doit être validée exactement une fois.")
display(pd.DataFrame(summary))
print("Vérifications réussies : aucune fuite de patients.")


## 4 — Entraînement des cinq folds
**Cette cellule lance les entraînements.** Chaque fold repart du checkpoint préentraîné. `best.pt` est réévalué sur son pli de validation. Un résultat par fold est sauvegardé immédiatement. Si une exécution est interrompue, conserver son dossier et choisir un nouveau nom d’expérience pour recommencer ; ce notebook ne reprend pas les entraînements.

In [ ]:
if not splits or len(splits) != 5:
    raise RuntimeError("Exécuter les vérifications avant l’entraînement.")
if RUN_ROOT.exists():
    raise FileExistsError("Choisir un nouveau nom EXPERIMENT : le dossier existe déjà.")
RUN_ROOT.mkdir(parents=True)
versions = {name: importlib.metadata.version(name) for name in
            ("torch", "torchvision", "ultralytics", "numpy", "opencv-python", "pyyaml", "pandas")}
manifest = dict(model=INITIAL_WEIGHTS, settings=settings, runtime=RUNTIME,
                versions=versions, python=sys.version, os=platform.platform())
(RUN_ROOT / "environment.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
records = []
for fold, descriptor in enumerate(FOLD_YAMLS, 1):
    print(f"\n===== FOLD {fold}/5 =====")
    started = time.perf_counter()
    model = YOLO(INITIAL_WEIGHTS)  # fresh pretrained weights, never previous fold weights
    model.train(data=str(descriptor.resolve()), cfg=str(CONFIG), **RUNTIME,
                project=str(RUN_ROOT), name=f"fold_{fold}", exist_ok=False)
    best = Path(model.trainer.save_dir) / "weights" / "best.pt"
    if not best.is_file():
        raise FileNotFoundError(best)
    history = pd.read_csv(best.parent.parent / "results.csv")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    evaluator = YOLO(str(best))
    metrics = evaluator.val(data=str(descriptor.resolve()), split="val",
                            imgsz=settings.get("imgsz", 512), device=0, batch=RUNTIME["batch"],
                            workers=0, plots=False, project=str(RUN_ROOT),
                            name=f"evaluation_{fold}", exist_ok=False)
    record = dict(fold=fold, mask_mAP50=float(metrics.seg.map50),
                  mask_mAP50_95=float(metrics.seg.map),
                  epochs_completed=len(history), minutes=(time.perf_counter() - started) / 60,
                  best_checkpoint=str(best.relative_to(ROOT)))
    records.append(record)
    pd.DataFrame(records).to_csv(RUN_ROOT / "fold_metrics.csv", index=False)
    print(record)
    del evaluator, metrics
    gc.collect()
    torch.cuda.empty_cache()
print("Les cinq entraînements et évaluations sont terminés.")


## 5 — Bilan
Moyennes non pondérées et écarts-types entre folds : ils décrivent les résultats de validation croisée, pas un intervalle de confiance ni une évaluation sur un test indépendant. Dice/IoU par patient ne sont pas calculés ici.

In [ ]:
results = pd.read_csv(RUN_ROOT / "fold_metrics.csv")
if sorted(results.fold.tolist()) != [1, 2, 3, 4, 5]:
    raise ValueError("Résultats incomplets : les cinq folds sont requis pour le bilan.")
display(results)
aggregate = results[["mask_mAP50", "mask_mAP50_95"]].agg(["mean", "std"])
display(aggregate)
aggregate.to_csv(RUN_ROOT / "summary_metrics.csv")
print(f"Durée totale entraînement + évaluation : {results.minutes.sum():.1f} minutes")


## 6 — Inspection visuelle hors entraînement
Deux images fixes par fold : référence PNG et prédiction du modèle qui n’a pas vu leur patient. Le seuil 0,25 est fixé pour l’affichage, il n’est pas optimisé sur ces exemples. La sélection visuelle ne remplace pas le bilan sur toutes les images.

In [ ]:
for row in results.itertuples():
    inspector = YOLO(str(ROOT / row.best_checkpoint))
    examples = sorted(splits[row.fold]["val"])[:2]
    for image in examples:
        prediction = inspector.predict(source=str(image), imgsz=settings.get("imgsz", 512),
                                       device=0, conf=0.25, save=False, verbose=False)[0]
        reference = cv2.imread(str(DATASET / "masks" / f"{image.stem}_mask.png"), cv2.IMREAD_GRAYSCALE)
        if reference is None:
            raise ValueError(f"Masque illisible : {image.name}")
        reference = reference > (0 if reference.max() <= 1 else 127)
        rgb = cv2.cvtColor(prediction.orig_img, cv2.COLOR_BGR2RGB)
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(rgb)
        if reference.any():
            axes[0].contour(reference.astype(float), levels=[0.5], colors="lime", linewidths=1)
        axes[0].set_title("Contour de référence PNG")
        axes[1].imshow(cv2.cvtColor(prediction.plot(), cv2.COLOR_BGR2RGB))
        axes[1].set_title("Prédiction YOLO")
        for axis in axes:
            axis.axis("off")
        fig.suptitle(f"Fold {row.fold} — {image.name}")
        plt.tight_layout()
        plt.show()
        plt.close(fig)
    del inspector
    gc.collect()
    torch.cuda.empty_cache()
